# Inter-Process Communication

---

## IPC

Processes are isolated from one other, but can have resources shared between them. So far, we have only covered signals as a way to communicate between processes. But this is clunky, and we can't use signals to reliably pass along data.

There are two main methods for effective communication between processes:
- Shared Memory
- Message Passing (pipes, sockets)

---

## Anonymous Pipes

Pipes create a unidirectional data channel between two processes that allow them to send messages to each other. Anonymous pipes flow through the operating system kernel; one end is writable, and anything written there will appear at the readable end of the pipe.

They are interacted with like files; when the OS opens a pipe, it provides two file descriptors for each end of the pipe.
```c
int pipe(int pipefd[2]);
```
`pipefd[0]` stores the file descriptor of the read end, and `pipefd[1]` stores the file descriptor of the write end.

This facilitates synchronous communication; there will be blocking calls when the pipe's buffer is empty or full. This way we can avoid problems with race conditions.


### Parent and Child Communication

It is standard to use unnamed pipes between parent and child processes, because we can set up the pipe before forking, and have the created file descriptors accessible to both processes (`fork` preserves the file descriptor table).
```c
// TODO: piping before forking
```
Bidirectional communication requires two pipes (i.e. 4 file descriptors): one for parent-to-child, and one for child-to-parent.

---

## Named Pipes

A named pipe, also known as a FIFO (first-in, first-out), is a special file analogous to pipes, but with a lifetime beyond the current process. Any external process can open the FIFO for reading and writing.

They are made with `mkfifo`:
```c
int mkfifo(const char* pathname, mode_t mode);
```
The `mode` is an octal number, similar to what the bash command `chmod` uses. To give read and write permissions for the FIFO to all users, set it to `0666`.

If `mkfifo` was successful, you can open the `pathname` for reading and writing in the same way you would with any other ordinary file. Note that like unnamed pipes, they are unidirectional, so two are needed for bidirectional communication.

---

## Asynchronous I/O

`select`, `poll` and `epoll` provide mechanisms for asynchronously monitoring multiple file descriptors, checking if I/O is possible on any of them without blocking. This is very crucial for servers or event-driven applications that need an efficient way to handle many connections concurrently.

### select

Waits for one or more file descriptors to become “ready” for reading, writing, or have error conditions.

To use, build three fd_set bitmasks (read, write, exception), add descriptors with `FD_SET`, then call:
```c
int n = select(max_fd + 1, &read_fds, &write_fds, &err_fds, &timeout);
```
After it returns, you test each set with `FD_ISSET` to see which descriptors are ready.

### poll

Available on many UNIX-like systems. It provides a simpler interface for monitoring file descriptors but has some limitations compared to epoll.

### epoll

Linux-specific. It uses an event-based interface to manage large numbers of file descriptors efficiently.

Start by creating an epoll instance:
```c
int epfd = epoll_create1(0);
```
Register each file descriptor with interest flags using `epoll_ctl`.
```c
struct epoll_event ev = { .events = EPOLLIN, .data.fd = sock };
epoll_ctl(epfd, EPOLL_CTL_ADD, sock, &ev);
```
Wait for events:
```c
struct epoll_event events[MAX_EVENTS];
int n = epoll_wait(epfd, events, MAX_EVENTS, timeout_ms);
```
The kernel then fills the events array with descriptors that have activity.

---

## Memory Mapping

Memory mapping is the second method of accomplishing IPC. Rather than file I/O through buffers, memory mapped files put the file's contents to memory, so that any changes made to that memory region changes the file.

The `<sys/mman.h>` function `mmap` allows for creation of a memory mapping, and `munmap` unmaps it. `mmap` is versitile, as it allows the programmer to specify how the memory is to be used; specifying `MAP_SHARED` when mmap is called provides other processes visibility to the same region of memory.
```c
void* mmap(void* addr, size_t length, int prot, int flags, int filedes, off_t offset);
int munmap(void* addr, size_t len);
```